# Proyek Klasifikasi Gambar: Pokemon Generation 1

Proyek ini bertujuan untuk mengklasifikasikan gambar Pokemon dari Generasi 1 menggunakan Convolutional Neural Network (CNN).  
Dataset yang digunakan: [Pokemon Images First Generation](https://www.kaggle.com/datasets/mikoajkolman/pokemon-images-first-generation17000-files)

## Kriteria Submission:
1. Dataset minimal 1000 gambar.
2. Tidak menggunakan dataset Rock-Paper-Scissors atau X-Ray.
3. Dataset dibagi menjadi Train, Test, dan Validation set.
4. Model menggunakan Sequential, Conv2D, dan Pooling Layer.
5. Akurasi Training dan Testing minimal 85%.
6. Plot Akurasi dan Loss.
7. Simpan model ke format SavedModel, TF-Lite, dan TFJS.

## 1. Import Library

In [ ]:
import os
import zipfile
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import files

print(f"TensorFlow Version: {tf.__version__}")

## 2. Persiapan Dataset (Google Colab)

In [ ]:
# Install Kaggle jika belum ada
!pip install -q kaggle

# Upload kaggle.json
print("Silakan upload file kaggle.json Anda untuk mengunduh dataset.")
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Token Kaggle berhasil dikonfigurasi.")
else:
    print("File kaggle.json tidak ditemukan!")

In [ ]:
# Download Dataset
dataset_name = 'mikoajkolman/pokemon-images-first-generation17000-files'
!kaggle datasets download -d {dataset_name}

# Ekstrak Dataset
local_zip = 'pokemon-images-first-generation17000-files.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('dataset')
zip_ref.close()

print("Dataset berhasil diekstrak.")

In [ ]:
# Verifikasi isi dataset
import pathlib
base_dir = pathlib.Path('dataset/dataset')
if not base_dir.exists():
    base_dir = pathlib.Path('dataset')

image_count = len(list(base_dir.glob('*/*.jpg'))) + len(list(base_dir.glob('*/*.png'))) + len(list(base_dir.glob('*/*.jpeg')))
print(f"Total gambar ditemukan: {image_count}")

if image_count < 10000:
    print("WARNING: Dataset kurang dari 10000 gambar!")
else:
    print("Dataset memenuhi kriteria minimal 10000 gambar.")

## 3. Data Preprocessing & Augmentasi

In [ ]:
# Split dataset: 80% Train, 20% Validation

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    shear_range=0.2,
    zoom_range=0.2,
    fill_mode='nearest',
    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

BATCH_SIZE = 32
TARGET_SIZE = (150, 150)

# Train Generator
train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Validation Generator
validation_generator = test_datagen.flow_from_directory(
    base_dir,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

## 4. Membangun Model CNN (Sequential)

In [ ]:
model = tf.keras.models.Sequential([
    # Conv 1
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    
    # Conv 2
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    
    # Conv 3
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    
    # Conv 4
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(train_generator.num_classes, activation='softmax')
])

model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## 5. Training Model dengan Callbacks

In [ ]:
# Callback untuk menghentikan training jika akurasi sudah tinggi
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if(logs.get('accuracy') > 0.96 and logs.get('val_accuracy') > 0.96):
            print("\nAkurasi telah mencapai >96%!")
            self.model.stop_training = True

callbacks = [
    myCallback(),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch = train_generator.samples // BATCH_SIZE,
    validation_data = validation_generator,
    validation_steps = validation_generator.samples // BATCH_SIZE,
    epochs = 40,
    callbacks = callbacks,
    verbose = 1
)

## 6. Plot Akurasi dan Loss

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, acc, 'r', label='Training Accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend(loc=0)

plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'r', label='Training Loss')
plt.plot(epochs, val_loss, 'b', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.legend(loc=0)

plt.show()

## 7. Convert & Save Model

In [ ]:
# 1. SavedModel
export_dir = 'submission/saved_model/'
tf.saved_model.save(model, export_dir)
print(f"Model SavedModel disimpan di: {export_dir}")

# 2. TF-Lite
converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
tflite_model = converter.convert()

tflite_path = 'submission/tflite/model.tflite'
os.makedirs('submission/tflite', exist_ok=True)
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"Model TF-Lite disimpan di: {tflite_path}")

# Simpan Labels
labels_path = 'submission/tflite/label.txt'
class_names = sorted(train_generator.class_indices.keys())
with open(labels_path, 'w') as f:
    f.write('\n'.join(class_names))
print(f"Label disimpan di: {labels_path}")

In [ ]:
# 3. TFJS
!pip install tensorflowjs
tfjs_path = 'submission/tfjs_model'
os.makedirs(tfjs_path, exist_ok=True)

!tensorflowjs_converter --input_format=keras_saved_model {export_dir} {tfjs_path}
print(f"Model TFJS disimpan di: {tfjs_path}")

In [ ]:
# Prepare Submission Folder Content
requirements_content = """tensorflow
tensorflowjs
matplotlib
pandas
numpy
kaggle
scipy
pillow"""

readme_content = """# Proyek Klasifikasi Gambar Pokemon

Proyek ini adalah submission untuk tugas akhir klasifikasi gambar.
Tujuan: Mengklasifikasikan Pokemon Generasi 1 menggunakan CNN.

## Model Architecture
Model menggunakan arsitektur Sequential CNN dengan 4 Layer Conv2D & MaxPooling2D.

## Cara Menggunakan
Ekstrak submission.zip. Folder ini berisi model dalam format SavedModel, TFLite, dan TFJS.
Jangan lupa sertakan notebook.ipynb ini dalam submission.
"""

with open('submission/requirements.txt', 'w') as f:
    f.write(requirements_content)

with open('submission/README.md', 'w') as f:
    f.write(readme_content)

print("File pelengkap berhasil dibuat di dalam folder submission.")

In [ ]:
# Download Folder Submission (Zipped)
!zip -r submission.zip submission
files.download('submission.zip')